<a href="https://colab.research.google.com/github/tommbk50-hub/Web-Designs/blob/main/KDDWebModelsPrep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install trimesh[easy]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.5/751.5 kB 13.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 28.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.5/14.5 MB 77.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [pycollada]


In [13]:
import trimesh
import numpy as np
import os
import glob
from trimesh.visual import TextureVisuals
from trimesh.visual.material import PBRMaterial

# 1. Dynamically scan /content for any .stl (or .STL) files
content_dir = "/content"
stl_files = glob.glob(os.path.join(content_dir, "*.stl")) + glob.glob(os.path.join(content_dir, "*.STL"))
stl_files = sorted(list(set(stl_files)))

print(f"Detected {len(stl_files)} STL file(s) in {content_dir}:")
for f in stl_files:
    print(f" - {os.path.basename(f)}")

# Heuristic to separate teeth files from structural files based on filename
teeth_files = {}
structural_files = {}

for file_path in stl_files:
    filename = os.path.basename(file_path)
    lower_name = filename.lower()
    if "teeth" in lower_name:
        if "upper" in lower_name:
            teeth_files["Upper"] = file_path
        elif "lower" in lower_name:
            teeth_files["Lower"] = file_path
        else:
            teeth_files[filename.replace(".stl", "").replace(".STL", "")] = file_path
    else:
        node_name = filename.replace(".stl", "").replace(".STL", "").replace(" ", "_")
        structural_files[node_name] = file_path

scene = trimesh.Scene()
parts_added = 0
teeth_added = 0

# 2. Load structural files preserving their native world coordinates
for node_name, actual_path in structural_files.items():
    if os.path.exists(actual_path):
        try:
            mesh = trimesh.load(actual_path, force='mesh')
            scene.add_geometry(mesh, node_name=node_name, geom_name=node_name)
            parts_added += 1
            print(f"Successfully loaded structural part: {node_name}")
        except Exception as e:
            print(f"Error loading structural part {node_name}: {e}")

# 3. Load teeth, split, filter, and sort
for jaw_type, actual_path in teeth_files.items():
    if os.path.exists(actual_path):
        try:
            teeth_mesh = trimesh.load(actual_path, force='mesh')
            shells = teeth_mesh.split(only_watertight=False)
            valid_teeth = [shell for shell in shells if len(shell.vertices) >= 30]
            valid_teeth.sort(key=lambda m: m.centroid[0])

            for i, tooth in enumerate(valid_teeth):
                node_name = f"Tooth_{jaw_type}_{i+1:02d}"
                scene.add_geometry(tooth, node_name=node_name, geom_name=node_name)
                teeth_added += 1

            print(f"Processed {len(valid_teeth)} valid teeth from '{os.path.basename(actual_path)}'.")
        except Exception as e:
            print(f"Error processing teeth file {os.path.basename(actual_path)}: {e}")

# Inside your Colab script, right before exporting the scene:
for i, geometry in enumerate(scene.geometry.values()):
    # Create a unique material for each tooth so they can be edited individually
    if hasattr(geometry.visual, 'material'):
        geometry.visual.material.name = f"ToothMaterial_{i}"
    else:
        mat = PBRMaterial(name=f"ToothMaterial_{i}")
        geometry.visual = TextureVisuals(material=mat)

# 4. Export the final combined scene
output_glb_path = "upper_mouth_model.glb"
scene.export(output_glb_path)

# 5. Print a summary
print("\n--- Assembly Summary ---")
print(f"Structural parts added: {parts_added}")
print(f"Individual teeth added: {teeth_added}")
print(f"Total items in the scene: {len(scene.geometry)}")
print(f"Final model exported to: {output_glb_path}")

Detected 1 STL file(s) in /content:
 - Upper+Teeth.stl
Processed 14 valid teeth from 'Upper+Teeth.stl'.

--- Assembly Summary ---
Structural parts added: 0
Individual teeth added: 14
Total items in the scene: 14
Final model exported to: upper_mouth_model.glb


In [14]:
import base64
import os
from IPython.display import HTML, display

glb_path = '/content/upper_mouth_model.glb'

if os.path.exists(glb_path):
    # Read the GLB file as binary
    with open(glb_path, 'rb') as f:
        glb_data = f.read()

    # Encode to base64
    b64_data = base64.b64encode(glb_data).decode('utf-8')
    data_uri = f"data:model/gltf-binary;base64,{b64_data}"

    # Use direct src with dark background and tight bounds
    html_code = f"""
    <script type="module" src="https://unpkg.com/@google/model-viewer/dist/model-viewer.min.js"></script>
    <model-viewer
        src="{data_uri}"
        auto-rotate
        camera-controls
        shadow-intensity="2"
        exposure="1.2"
        bounds="tight"
        camera-target="auto"
        style="width: 100%; height: 600px; background-color: #222222; border-radius: 8px;">
    </model-viewer>
    """

    display(HTML(html_code))
else:
    print(f"Error: File '{glb_path}' not found.")

In [16]:
import trimesh
import numpy as np
import os
import glob
from trimesh.visual import TextureVisuals
from trimesh.visual.material import PBRMaterial

# 1. Dynamically scan /content for any .stl (or .STL) files
content_dir = "/content"
stl_files = glob.glob(os.path.join(content_dir, "*.stl")) + glob.glob(os.path.join(content_dir, "*.STL"))
stl_files = sorted(list(set(stl_files)))

print(f"Detected {len(stl_files)} STL file(s) in {content_dir}:")
for f in stl_files:
    print(f" - {os.path.basename(f)}")

for file_path in stl_files:
    filename = os.path.basename(file_path)
    base_name = os.path.splitext(filename)[0]
    lower_name = filename.lower()

    scene = trimesh.Scene()
    print(f"\nProcessing {filename}...")

    if "teeth" in lower_name:
        try:
            teeth_mesh = trimesh.load(file_path, force='mesh')
            shells = teeth_mesh.split(only_watertight=False)
            valid_teeth = [shell for shell in shells if len(shell.vertices) >= 30]
            valid_teeth.sort(key=lambda m: m.centroid[0])

            for i, tooth in enumerate(valid_teeth):
                node_name = f"Tooth_{i+1:02d}"
                scene.add_geometry(tooth, node_name=node_name, geom_name=node_name)

            print(f"Processed {len(valid_teeth)} valid teeth from '{filename}'.")

            # Apply unique materials to each tooth
            for i, geometry in enumerate(scene.geometry.values()):
                if hasattr(geometry.visual, 'material'):
                    geometry.visual.material.name = f"ToothMaterial_{i}"
                else:
                    mat = PBRMaterial(name=f"ToothMaterial_{i}")
                    geometry.visual = TextureVisuals(material=mat)

        except Exception as e:
            print(f"Error processing teeth file {filename}: {e}")
            continue
    else:
        try:
            mesh = trimesh.load(file_path, force='mesh')
            scene.add_geometry(mesh, node_name=base_name, geom_name=base_name)
            print(f"Successfully loaded structural part: {filename}")

            # Assign a default material to the structural part
            for i, geometry in enumerate(scene.geometry.values()):
                if not hasattr(geometry.visual, 'material'):
                    mat = PBRMaterial(name=f"{base_name}_Material")
                    geometry.visual = TextureVisuals(material=mat)

        except Exception as e:
            print(f"Error loading structural part {filename}: {e}")
            continue

    # Export the scene to GLB
    output_glb_path = os.path.join(content_dir, f"{base_name}.glb")
    scene.export(output_glb_path)
    print(f"Exported to: {output_glb_path}")

print("\n--- Processing Complete ---")

Detected 4 STL file(s) in /content:
 - Lower+Stümpfe.stl
 - Lower+Teeth.stl
 - Upper+Stümpfe.stl
 - Upper+Teeth.stl

Processing Lower+Stümpfe.stl...
Successfully loaded structural part: Lower+Stümpfe.stl
Exported to: /content/Lower+Stümpfe.glb

Processing Lower+Teeth.stl...
Processed 14 valid teeth from 'Lower+Teeth.stl'.
Exported to: /content/Lower+Teeth.glb

Processing Upper+Stümpfe.stl...
Successfully loaded structural part: Upper+Stümpfe.stl
Exported to: /content/Upper+Stümpfe.glb

Processing Upper+Teeth.stl...
Processed 14 valid teeth from 'Upper+Teeth.stl'.
Exported to: /content/Upper+Teeth.glb

--- Processing Complete ---
